# Classificação de Cobertura do Solo
**Objetivo:** Construir um pipeline de Machine Learning capaz de maximizar o *Recall* de classes minoritárias (como Solo e Piscina) sem sacrificar a precisão global, utilizando otimização de hiperparâmetros.


In [ ]:
# 1. Importações Essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Algoritmos Base e Ensembles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier

pd.set_option('display.max_columns', None)


## 1. Carregamento e Preparação dos Dados


In [ ]:
# Carregando os datasets
treino = pd.read_csv("/content/training.csv") # Substitua pelo caminho real
teste = pd.read_csv("/content/testing.csv")   # Substitua pelo caminho real

# Padronização do nome das classes (minúsculo e sem espaços)
treino['class'] = treino['class'].astype(str).str.strip().str.lower()
teste['class'] = teste['class'].astype(str).str.strip().str.lower()

print(f"Shape do Treino: {treino.shape}")
print("\nDistribuição das Classes no Treino:")
print(treino['class'].value_counts())


## 2. Separação de Features e Codificação do Target


In [ ]:
# Separando Features e Target
X_treino = treino.drop(columns='class')
y_treino = treino['class']

X_teste = teste.drop(columns='class')
y_teste = teste['class']

# Codificação do Target
le = LabelEncoder()
y_treino_encoded = le.fit_transform(y_treino)
y_teste_encoded = le.transform(y_teste)

# Mapeamento para referência
mapa_classes = {indice: classe for indice, classe in enumerate(le.classes_)}
print("Mapeamento de Classes:", mapa_classes)


## 3. Seleção do Modelo Base (Cross-Validation)
Teste de alguns algoritmos com pesos balanceados para encontrar a melhor arquitetura inicial.


In [ ]:
# Validador Estratificado
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Dicionário de Modelos para teste
modelos = {
    "Regressão Logística": Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(multi_class='multinomial', solver='lbfgs', class_weight='balanced', random_state=42, max_iter=1000))]),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "Balanced RF": BalancedRandomForestClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=0)
}

resultados_cv = []
print("Avaliando modelos...")
for nome, modelo in modelos.items():
    res = cross_validate(modelo, X_treino, y_treino_encoded, cv=skf, scoring=['accuracy', 'f1_macro'])
    resultados_cv.append({
        'Modelo': nome,
        'Acurácia Média': res['test_accuracy'].mean(),
        'F1-Macro Médio': res['test_f1_macro'].mean()
    })

df_resultados = pd.DataFrame(resultados_cv).sort_values(by='F1-Macro Médio', ascending=False).reset_index(drop=True)
display(df_resultados)


## 4. Otimização de Hiperparâmetros (Tuning - GridSearchCV)
O `BalancedRandomForestClassifier` mostrou-se promissor.


In [ ]:
# 1. Linha de Montagem
pipeline_brf = Pipeline(steps=[
    ('brf_model', BalancedRandomForestClassifier(random_state=42, n_jobs=-1))
])

# 2. O Menu de Hiperparâmetros
param_grid = {
    'brf_model__n_estimators': [100, 200, 300], 
    'brf_model__max_depth': [None, 10, 15, 20],      
    'brf_model__min_samples_split': [2, 5],      
    'brf_model__min_samples_leaf': [1, 2]         
}

# 3. Busca em Grade
grid_search = GridSearchCV(
    estimator=pipeline_brf,
    param_grid=param_grid,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

# 4. Treinamento
print("Iniciando GridSearchCV (isso pode levar alguns minutos)...")
grid_search.fit(X_treino, y_treino_encoded)

print("\n--- Melhores Hiperparâmetros Encontrados ---")
print(grid_search.best_params_)

# Salvando o modelo campeão
melhor_pipeline = grid_search.best_estimator_


## 5. Avaliação do Modelo Otimizado (Relatório e Matriz de Confusão)
Avaliação final no conjunto de teste.


In [ ]:
# Previsões com o modelo otimizado
previsoes_otimizadas = melhor_pipeline.predict(X_teste)

print("--- Relatório Final do Modelo Otimizado ---")
print(f"Acurácia Global: {accuracy_score(y_teste_encoded, previsoes_otimizadas):.2f}\n")
print(classification_report(y_teste_encoded, previsoes_otimizadas, target_names=le.classes_))

# Matriz de Confusão
cm = confusion_matrix(y_teste_encoded, previsoes_otimizadas)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Matriz de Confusão - BRF Otimizado", fontsize=14, pad=15)
plt.ylabel("Classe Real", fontsize=12)
plt.xlabel("Classe Prevista", fontsize=12)
plt.tight_layout()
plt.show()


## 6. Interpretabilidade: Feature Importance
Quais características o modelo otimizado considerou mais importantes após a poda?


In [ ]:
# Extraindo o modelo de dentro do pipeline
modelo_rf_final = melhor_pipeline.named_steps['brf_model']

# DataFrame de importância
importancias = pd.DataFrame({
    'Feature': X_treino.columns,
    'Importância': modelo_rf_final.feature_importances_
}).sort_values(ascending=False, by='Importância')

# Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='Importância', y='Feature', hue='Feature', data=importancias.head(20), palette='viridis', legend=False)
plt.title("Top 20 Features Mais Importantes - BRF Otimizado", fontsize=14, pad=15)
plt.xlabel("Importância Relativa")
plt.ylabel("Features")
plt.tight_layout()
plt.show()


## 7. Exportação do Modelo (Deployment)
Salvando o pipeline para ser consumida em produção via API.


In [ ]:
# Exportando com joblib
nome_arquivo = 'urban_land_cover_model_brf.pkl'
joblib.dump(melhor_pipeline, nome_arquivo)
